# Preparing a Dataset for Analysis

Before any analysis can begin, the underlying data must be **understood, cleaned, and shaped** into a form that supports reliable conclusions. Skipping this step is one of the most common causes of incorrect results, misleading dashboards, and wasted effort.

> **Note:** This module uses **SQL** throughout. SQL is the most cross-compute compatible language — it runs on classic clusters, serverless SQL warehouses, and virtually every database engine, making the techniques here portable regardless of your platform.

> **Data disclaimer:** All data used in this module is **entirely synthetic** and does not represent any real schools, pupils, or individuals. No real data has been used.

> **Fiddle:** Some of these notebooks follow the pattern of showing you a manual way to achieve something which is relatively straight forward. They then show you a *dynamic* way of doing the same task that may be more abstract and confusing to begin with, but in the long run makes the task more automated. 
**It is not necessary to fully understand these dynamic scripts**, however they are very valuable techniques that can be very useful in the long run so if you have the time to play around with the code, ask the AI assistant questions about it, and generally improve your understanding of the concepts it is encouraged to do so.

## Why Does Messy Data Matter?

It is tempting to skip straight to analysis, but working with data that hasn’t been cleaned and prepared leads to **wrong answers that look right**. The examples below demonstrate how common data quality issues — duplicates, inconsistent labels, and fragmented sources — produce misleading results silently, with no error messages to warn you.

### The danger: no errors, just incorrect figures

Most data quality problems don’t cause your query to fail. They cause it to return a result that is **plausible but incorrect**. A headcount inflated by 25%, a region split into three groups instead of one, or a pupil quietly dropped from a join — these are the kinds of issues that erode trust in reporting and lead to poor decisions.

### Problem 1: Duplicates inflate your counts

If the same record appears more than once, every `COUNT`, `SUM`, and `AVG` is affected. The `pupils_autumn_2024` table in our messy dataset contains both exact duplicates and natural key conflicts. The query below groups by school and compares the raw row count to the distinct pupil count — any difference is inflation caused by duplicates.

In [0]:
-- The pupils_autumn_2024 table contains duplicate rows
-- Compare raw count to distinct pupil count per school to reveal inflation
SELECT
  school_urn
  ,COUNT(*) AS reported_headcount
  ,COUNT(DISTINCT pupil_id) AS actual_headcount
  ,COUNT(*) - COUNT(DISTINCT pupil_id) AS inflation
FROM catalog_40_copper_analyst_training.bronze.pupils_autumn_2024
GROUP BY school_urn
ORDER BY inflation DESC;

### Problem 2: Inconsistent labels fragment your groups

If the same value is recorded in different ways — different casing, abbreviations, or placeholder values — a `GROUP BY` treats each variant as a separate category. The `schools_autumn_2024` table records school type inconsistently: `'Academy'`, `'academy'`, `'Acad'`, and `'N/A'` all appear. The query below shows how what should be two or three groups becomes six.

In [0]:
-- The schools_autumn_2024 table has inconsistent school_type values
-- GROUP BY treats each variant as a separate category
SELECT
  school_type
  ,COUNT(*) AS school_count
FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2024
GROUP BY school_type
ORDER BY school_type;

### Problem 3: Fragmented data loses records silently

When reference data is incomplete, joins can **silently drop rows**. The `pupils_autumn_2024` table contains a pupil (P019) enrolled at school URN 999999 — a school that doesn’t exist in `schools_autumn_2024`. An inner join quietly removes that pupil from the results with no error or warning.

In [0]:
-- How many distinct pupils exist vs how many survive an inner join to schools?
-- The difference is pupils silently lost due to missing school references
SELECT
  'Total distinct pupils' AS metric,
  COUNT(DISTINCT pupil_id) AS count
FROM catalog_40_copper_analyst_training.bronze.pupils_autumn_2024

UNION ALL

SELECT
  'Pupils after INNER JOIN to schools',
  COUNT(DISTINCT p.pupil_id)
FROM catalog_40_copper_analyst_training.bronze.pupils_autumn_2024 p
INNER JOIN catalog_40_copper_analyst_training.bronze.schools_autumn_2024 s
  ON p.school_urn = s.school_urn;

## The Benefits of Data Preparation

Investing time in data preparation makes every stage of the analytical process significantly easier. The benefits go well beyond "cleaner data".

### Accuracy

The most fundamental benefit. When duplicates are removed, labels are standardised, and keys are validated, you can **trust the numbers**. Stakeholders make decisions based on your outputs — if a headcount is inflated by 25% because of duplicates (as in the example above), that directly affects funding, staffing, and planning.

### Consistency

By cleaning data **once at the start**, you guarantee that every downstream query, report, and dashboard applies the **same transformations in the same way**. Without this, different analysts (or even the same analyst on different days) may make slightly different cleaning decisions — one trims whitespace but not casing, another maps `'Acad'` to `'Academy'` but misses `'academy'`. The result is inconsistent outputs from the same source data, which is difficult to diagnose and erodes stakeholder confidence. A single, systematic preparation step eliminates this risk entirely.

### Documentation

The process of preparing data forces you to **understand and record** what the data contains, where it came from, and what decisions were made. This creates a trail that:
* Makes your work **auditable** — others can see what was included, excluded, and why
* Makes handovers easier — a colleague can pick up your work without reverse-engineering it
* Protects you — if a number is ever questioned, you can point to a documented rationale

### Time saving

It feels slower upfront, but preparation **saves far more time than it costs**. Without it, you’ll spend hours debugging unexpected results, re-running queries with different filters to understand anomalies, and answering questions from stakeholders about why numbers don’t add up. A well-prepared dataset can be queried confidently from the start.

### Cost saving

In cloud environments, compute is billed by usage. Messy data leads to:
* **Larger datasets** — duplicates mean more rows to scan, store, and process
* **Repeated work** — every downstream query re-applies the same cleaning logic, spending compute costs each time
* **Wider scans** — unstandardised fields prevent effective filtering and partitioning, forcing full table scans

Preparing data once and writing it to a clean table means every subsequent query is faster and cheaper.

### Reproducibility

When preparation steps are codified in a notebook or pipeline, the same process can be **re-run on new data** and produce consistent results. This is essential for periodic reporting — if you clean data manually or ad hoc, each refresh introduces the risk of doing it differently.

### Collaboration

A well-prepared dataset with standardised field names, consistent labels, and documented grain is **immediately usable by anyone** on the team. Without preparation, each analyst must first understand the quirks of the raw data before they can do anything useful — multiplying the wasted effort across the team.

### Fitness for purpose

Different analyses require data at different grains and with different levels of completeness. Preparation is where you make deliberate decisions about what the data should look like for a specific use case. A dataset prepared for school-level reporting will look different from one prepared for pupil-level analysis — and that’s by design, not accident.

## Summary

Preparing data for analysis is not optional — it's the foundation that everything else depends on. Before writing a single analytical query, work through these steps:

1. **Combine your sources** — use `UNION ALL` with schema alignment, tag each source, and validate row counts
2. **Reconcile schema drift** — unify columns that were renamed between snapshots, extract temporal dimensions
3. **Remove exact duplicates** — a mechanical first step that requires no domain knowledge but clears the noise for everything that follows
4. **Understand the grain** — check cardinality, identify keys, and validate foreign key relationships
5. **Resolve natural key conflicts** — where the same key has different values, apply a clear business rule to choose which row survives
6. **Standardise relentlessly** — normalise case, trim whitespace, map variant labels, and unify nulls

### Notebook roadmap

The remaining notebooks in this module walk through each step using the synthetic schools and pupils data:

| Notebook | What you'll learn |
| --- | --- |
| **02 — Medallion Architecture** | The bronze → silver → gold pattern and how the DfE implements it |
| **03 — Combining Snapshots** | Combine termly bronze extracts into `silver.schools_combined` and `silver.pupils_combined` |
| **04 — Reconciling Schema Drift** | Unify renamed columns and extract `term` and `year` into `silver.schools_reconciled` and `silver.pupils_reconciled` |
| **05 — Understanding Your Data** | Remove exact duplicates, then check cardinality, keys, and referential integrity |
| **06 — Removing Duplicates** | Resolve natural key conflicts using business rules and `ROW_NUMBER` |
| **07 — Standardising Fields and Labels** | Clean values: casing, whitespace, abbreviations, dates |
| **08 — Building the Gold Layer** | Apply all steps end-to-end and write two business-ready tables to gold |